# 🚀 Production Agents

**Deploy agents in production**

---

## 📋 Overview

**What you'll learn:**
- Production agent architecture
- Error handling and retry logic
- Monitoring and logging
- Cost optimization
- Best practices

**Time estimate:** ⏱️ 55 minutes | **Difficulty:** 🔴 Advanced

---

In [ ]:
from openai import OpenAI
import os
import json
import time
import logging
from typing import Dict, List, Optional
from datetime import datetime

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 Production Challenges

### Development vs Production:

**Development:**
```python
def simple_agent(query):
    response = llm.call(query)
    return response

✅ Works for demos
❌ Fails in production
```

**Production:**
```python
def production_agent(query):
    # Validate input
    # Rate limiting
    # Retry logic
    # Error handling
    # Logging
    # Monitoring
    # Cost tracking
    # Timeout handling
    # Response validation
    return response

✅ Production-ready
```

### Key Production Requirements:

1. **Reliability** - Must handle errors gracefully
2. **Observability** - Know what's happening
3. **Performance** - Meet latency SLAs
4. **Cost** - Stay within budget
5. **Safety** - Prevent harmful outputs

## 🏗️ Production Agent Architecture

In [ ]:
class ProductionAgent:
    """Production-ready agent with monitoring, retries, and error handling."""
    
    def __init__(
        self,
        name: str,
        tools: List[Dict],
        max_iterations: int = 10,
        max_retries: int = 3,
        timeout: int = 60
    ):
        self.name = name
        self.tools = tools
        self.max_iterations = max_iterations
        self.max_retries = max_retries
        self.timeout = timeout
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        
        # Metrics
        self.metrics = {
            'total_requests': 0,
            'successful_requests': 0,
            'failed_requests': 0,
            'total_tokens': 0,
            'total_cost': 0.0,
            'avg_latency': 0.0
        }
    
    def run(self, query: str) -> Dict:
        """Run agent with full production safeguards."""
        
        start_time = time.time()
        self.metrics['total_requests'] += 1
        
        logger.info(f"Agent {self.name} received query: {query[:100]}...")
        
        try:
            # Validate input
            self._validate_input(query)
            
            # Run with timeout
            result = self._run_with_timeout(query)
            
            # Validate output
            self._validate_output(result)
            
            # Update metrics
            latency = time.time() - start_time
            self.metrics['successful_requests'] += 1
            self._update_metrics(result, latency)
            
            logger.info(f"Agent completed successfully in {latency:.2f}s")
            
            return {
                'success': True,
                'result': result,
                'latency': latency,
                'iterations': result.get('iterations', 0)
            }
            
        except Exception as e:
            self.metrics['failed_requests'] += 1
            logger.error(f"Agent failed: {str(e)}", exc_info=True)
            
            return {
                'success': False,
                'error': str(e),
                'latency': time.time() - start_time
            }
    
    def _validate_input(self, query: str):
        """Validate input query."""
        if not query or not query.strip():
            raise ValueError("Query cannot be empty")
        
        if len(query) > 10000:
            raise ValueError("Query too long (max 10000 chars)")
        
        # Check for malicious content
        malicious_patterns = ['DROP TABLE', 'DELETE FROM', '<script>']
        if any(pattern in query.upper() for pattern in malicious_patterns):
            raise ValueError("Potentially malicious query detected")
    
    def _run_with_timeout(self, query: str) -> Dict:
        """Run agent with timeout."""
        import signal
        
        def timeout_handler(signum, frame):
            raise TimeoutError(f"Agent exceeded timeout of {self.timeout}s")
        
        # Set timeout (Unix only)
        # signal.signal(signal.SIGALRM, timeout_handler)
        # signal.alarm(self.timeout)
        
        try:
            result = self._execute_agent(query)
            # signal.alarm(0)  # Cancel timeout
            return result
        except TimeoutError:
            # signal.alarm(0)
            raise
    
    def _execute_agent(self, query: str) -> Dict:
        """Core agent execution with retries."""
        
        messages = [{"role": "user", "content": query}]
        iterations = 0
        total_tokens = 0
        
        for iteration in range(self.max_iterations):
            iterations += 1
            
            # Call LLM with retries
            response = self._call_llm_with_retry(messages)
            
            total_tokens += response.usage.total_tokens
            response_message = response.choices[0].message
            
            # Check if done
            if not response_message.tool_calls:
                return {
                    'answer': response_message.content,
                    'iterations': iterations,
                    'total_tokens': total_tokens
                }
            
            # Execute tools
            messages.append(response_message)
            
            for tool_call in response_message.tool_calls:
                result = self._execute_tool_with_retry(tool_call)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })
        
        raise RuntimeError("Max iterations reached without completion")
    
    def _call_llm_with_retry(self, messages: List[Dict]):
        """Call LLM with exponential backoff retry."""
        
        for attempt in range(self.max_retries):
            try:
                return self.client.chat.completions.create(
                    model="gpt-4",
                    messages=messages,
                    tools=self.tools,
                    tool_choice="auto"
                )
            except Exception as e:
                if attempt == self.max_retries - 1:
                    raise
                
                wait_time = 2 ** attempt
                logger.warning(f"LLM call failed, retrying in {wait_time}s: {e}")
                time.sleep(wait_time)
    
    def _execute_tool_with_retry(self, tool_call) -> Dict:
        """Execute tool with retry logic."""
        
        for attempt in range(self.max_retries):
            try:
                # Mock tool execution
                return {"status": "success", "result": "Tool executed"}
            except Exception as e:
                if attempt == self.max_retries - 1:
                    return {"status": "error", "error": str(e)}
                
                wait_time = 2 ** attempt
                logger.warning(f"Tool execution failed, retrying in {wait_time}s")
                time.sleep(wait_time)
    
    def _validate_output(self, result: Dict):
        """Validate agent output."""
        if 'answer' not in result:
            raise ValueError("Agent did not produce an answer")
        
        # Check for empty response
        if not result['answer'] or not result['answer'].strip():
            raise ValueError("Agent produced empty answer")
    
    def _update_metrics(self, result: Dict, latency: float):
        """Update agent metrics."""
        self.metrics['total_tokens'] += result.get('total_tokens', 0)
        self.metrics['total_cost'] += self._calculate_cost(result.get('total_tokens', 0))
        
        # Update average latency
        total = self.metrics['successful_requests']
        prev_avg = self.metrics['avg_latency']
        self.metrics['avg_latency'] = (prev_avg * (total - 1) + latency) / total
    
    def _calculate_cost(self, tokens: int) -> float:
        """Calculate cost based on tokens."""
        # GPT-4 pricing (approximate)
        cost_per_1k_tokens = 0.03
        return (tokens / 1000) * cost_per_1k_tokens
    
    def get_metrics(self) -> Dict:
        """Get agent metrics."""
        return {
            **self.metrics,
            'success_rate': (
                self.metrics['successful_requests'] / max(self.metrics['total_requests'], 1)
            ) * 100
        }

# Example usage
agent = ProductionAgent(
    name="ProductionAgent",
    tools=[],
    max_iterations=5,
    max_retries=3,
    timeout=30
)

print("🚀 Production Agent initialized")
print(f"   Max iterations: {agent.max_iterations}")
print(f"   Max retries: {agent.max_retries}")
print(f"   Timeout: {agent.timeout}s")

## 📊 Monitoring Dashboard

In [ ]:
class AgentMonitor:
    """Monitor agent performance."""
    
    def __init__(self):
        self.agents = {}
    
    def register_agent(self, agent: ProductionAgent):
        """Register an agent for monitoring."""
        self.agents[agent.name] = agent
    
    def get_dashboard(self) -> str:
        """Get monitoring dashboard."""
        
        dashboard = "\n📊 Agent Monitoring Dashboard\n"
        dashboard += "=" * 60 + "\n"
        
        for name, agent in self.agents.items():
            metrics = agent.get_metrics()
            
            dashboard += f"\n🤖 {name}:\n"
            dashboard += f"  Total requests: {metrics['total_requests']}\n"
            dashboard += f"  Success rate: {metrics['success_rate']:.1f}%\n"
            dashboard += f"  Avg latency: {metrics['avg_latency']:.2f}s\n"
            dashboard += f"  Total tokens: {metrics['total_tokens']:,}\n"
            dashboard += f"  Total cost: ${metrics['total_cost']:.4f}\n"
            
            # Alerts
            if metrics['success_rate'] < 95:
                dashboard += f"  ⚠️  Low success rate!\n"
            if metrics['avg_latency'] > 5:
                dashboard += f"  ⚠️  High latency!\n"
        
        return dashboard
    
    def check_health(self) -> Dict:
        """Check overall system health."""
        
        total_requests = sum(a.metrics['total_requests'] for a in self.agents.values())
        total_success = sum(a.metrics['successful_requests'] for a in self.agents.values())
        
        if total_requests == 0:
            return {'healthy': True, 'message': 'No requests yet'}
        
        success_rate = (total_success / total_requests) * 100
        
        return {
            'healthy': success_rate >= 95,
            'success_rate': success_rate,
            'total_requests': total_requests
        }

# Example
monitor = AgentMonitor()
monitor.register_agent(agent)

print(monitor.get_dashboard())

## ✅ Summary

### Production Checklist:

**✅ Error Handling:**
- Retry logic with exponential backoff
- Graceful degradation
- Timeout handling
- Input/output validation

**✅ Observability:**
- Structured logging
- Metrics tracking
- Performance monitoring
- Cost tracking

**✅ Reliability:**
- Max iterations limit
- Circuit breakers
- Fallback mechanisms
- Health checks

**✅ Performance:**
- Caching
- Parallel execution
- Request batching
- Rate limiting

**✅ Security:**
- Input sanitization
- Output filtering
- API key management
- Access control

### Best Practices:

**1. Always Retry**
```python
for attempt in range(max_retries):
    try:
        return llm.call()
    except:
        if attempt == max_retries - 1:
            raise
        time.sleep(2 ** attempt)  # Exponential backoff
```

**2. Log Everything**
```python
logger.info(f"Agent started: {query}")
logger.debug(f"Iteration {i}: {action}")
logger.error(f"Agent failed: {error}")
```

**3. Track Metrics**
```python
metrics = {
    'requests': 0,
    'errors': 0,
    'latency': [],
    'tokens': 0,
    'cost': 0.0
}
```

**4. Set Limits**
```python
MAX_ITERATIONS = 10
MAX_RETRIES = 3
TIMEOUT = 60
MAX_TOKENS = 4000
```

### Congratulations! 🎉

You've completed the Agents & Tools module!

**Next module:** `08_production_apis/` - Building production APIs